<a href="https://colab.research.google.com/github/amirgroup-codes/ProGenMech/blob/main/ProGenMech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://raw.githubusercontent.com/amirgroup-codes/ProGenMech/main/ProGenMech_Logo_Dark.svg">
    <img src="https://raw.githubusercontent.com/amirgroup-codes/ProGenMech/main/ProGenMech_Logo_Light.svg"
         alt="ProGenMech"
         width="60%">
  </picture>
</p>

# ProGenMech: Protein Circuit Tracing via Cross-layer Transcoders in ProGen3

ProGenMech is a framework for discovering computational circuits in the autoregressive protein language model ProGen3 using cross-layer transcoders (CLTs). This Colab notebook is designed to produce the files required for our [website](https://protmech.github.io/):
1. `activation_indices.json`
2. `seq.txt` (or `generation.fasta` for CLM tasks)
3. `top_activations.json`
4. `virtual_weights.json`

A link to the paper can be found [here](https://arxiv.org/abs/2602.12026). We additionally provide our [code](https://github.com/amirgroup-codes/ProGenMech), [models](https://huggingface.co/darintsui/ProGenMechModels), and [data](https://huggingface.co/datasets/darintsui/ProGenMechData).

Unlike ProtoMech (ESM2), ProGenMech uses ProGen3 with specialized dependencies (`external/progen3`, `megablocks`, `triton`, `flash-attn`). **Use a T4 GPU or better.** On Colab T4 (sm_75), ProGen3 is automatically reloaded with `moe_implementation='eager'` (no megablocks/Triton MoE) plus PyTorch RMSNorm and fallback attention; A100/L4 are faster.

---

In [ ]:
# @title 0. Check GPU status and install dependencies
import os
import sys
import pty
import shutil
import subprocess
import argparse

import torch
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    print(f"GPU Detected: {torch.cuda.get_device_name(0)} (sm_{cc_major}{cc_minor})")
    print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
    print(f"Python {sys.version.split()[0]}")
    if cc_major < 8:
        print("Note: T4 / legacy GPU detected. ProGen3 will use eager MoE + PyTorch fallbacks (no Triton).")
else:
    print("WARNING: No GPU detected. ProGen3 requires a GPU runtime.")
    print("----------------------------------------------------------------")
    print("TO FIX THIS:")
    print("1. Click 'Runtime' in the top menu.")
    print("2. Select 'Change runtime type'.")
    print("3. Under 'Hardware accelerator', select 'T4 GPU' (or better).")
    print("4. Click 'Save' and then re-run this cell.")
    print("----------------------------------------------------------------")

if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([argparse.Namespace])

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from huggingface_hub import hf_hub_download
import ipywidgets as widgets
from IPython.display import display

# Core Python dependencies
!pip install -q huggingface_hub pytorch-lightning pandas scipy einops "transformers>=4.42,<4.49" tqdm ipywidgets

# ProGen3-specific dependencies
!pip install --no-build-isolation -q "megablocks[gg]==0.7.0" grouped-gemm stanford-stk

# flash-attn: prefer a prebuilt wheel on Colab (source builds can hang for 30+ min)
import importlib

def install_flash_attn():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    import torch as _torch
    torch_mm = ".".join(_torch.__version__.split("+")[0].split(".")[:2])
    abi = "TRUE" if _torch.compiled_with_cxx11_abi() else "FALSE"
    wheel = (
        f"https://github.com/Dao-AILab/flash-attention/releases/download/"
        f"v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch{torch_mm}cxx11abi{abi}-"
        f"{py_tag}-{py_tag}-linux_x86_64.whl"
    )
    print(f"Trying prebuilt flash-attn wheel for torch {torch_mm}, {py_tag}, cxx11abi={abi}...")
    rc = os.system(f"pip install -q {wheel}")
    if rc != 0:
        print("Prebuilt wheel not found for this environment; falling back to source build (may take a while)...")
        os.system("MAX_JOBS=4 pip install -q flash-attn==2.7.4.post1 --no-build-isolation")

install_flash_attn()

# Verify the hard-required ProGen3 imports
importlib.import_module("megablocks")
importlib.import_module("flash_attn")
importlib.import_module("triton")
print("flash-attn / megablocks / triton imports OK")
print("Dependency installation complete.")

In [ ]:
# @title 1. Clone ProGenMech and download models and data
from huggingface_hub import hf_hub_download, snapshot_download

REPO_URL = "https://github.com/amirgroup-codes/ProGenMech.git"
BRANCH = "main"
TARGET_DIR = "/content/ProGenMech"

MODEL_REPO = "darintsui/ProGenMechModels"
DATA_REPO = "darintsui/ProGenMechData"
PROGEN3_MODEL = "Profluent-Bio/progen3-112m"
clt_filename = "ProGen3_CLT_L10_D4608/checkpoints/last.ckpt"
activations_filename = "top10_activations.pt"

# 1. Clean up or check for existing directory
if os.path.exists(TARGET_DIR):
    print(f"Directory {TARGET_DIR} already exists. Preparing for a fresh clone...")
    shutil.rmtree(TARGET_DIR)

# 2. Clone the repository (Public URL)
print(f"Cloning {REPO_URL}...")
result = os.system(f"git clone -q -b {BRANCH} {REPO_URL} {TARGET_DIR}")

if result == 0:
    print(f"Successfully cloned ProGenMech into {TARGET_DIR}")
else:
    raise RuntimeError("Failed to clone repository.")

# 3. Add to sys.path so imports work immediately
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)
PROGEN3_SRC = os.path.join(TARGET_DIR, "external", "progen3", "src")
if PROGEN3_SRC not in sys.path:
    sys.path.append(PROGEN3_SRC)

# 4. Install vendored ProGen3 package
PROGEN3_ROOT = os.path.join(TARGET_DIR, "external", "progen3")
if os.path.isdir(PROGEN3_ROOT):
    !pip install -q -e {PROGEN3_ROOT}
else:
    raise FileNotFoundError(f"Missing external/progen3 at {PROGEN3_ROOT}")

# 5. Download CLT checkpoint and reference activations
print("Downloading ProGenMech models and data...")
MODELS_DIR = os.path.join(TARGET_DIR, "models")
VISUALIZATION_DIR = os.path.join(TARGET_DIR, "visualization")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VISUALIZATION_DIR, exist_ok=True)

hf_hub_download(
    repo_id=MODEL_REPO,
    filename=clt_filename,
    repo_type="model",
    local_dir=MODELS_DIR,
)
hf_hub_download(
    repo_id=DATA_REPO,
    filename=activations_filename,
    repo_type="dataset",
    local_dir=VISUALIZATION_DIR,
)

clt_checkpoint = os.path.join(MODELS_DIR, clt_filename)
activations_pt = os.path.join(VISUALIZATION_DIR, activations_filename)

os.environ["CLT_CHECKPOINT"] = clt_checkpoint
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PROGEN3_MODEL"] = PROGEN3_MODEL

if not os.path.exists(clt_checkpoint):
    raise ValueError(f"CLT not found at {clt_checkpoint}, check download")
if not os.path.exists(activations_pt):
    raise ValueError(f"{activations_pt} not found, check download")

# 6. Prefetch ProGen3 base weights (required when loading the CLT checkpoint)
print(
    "\nPrefetching ProGen3 base weights.\n"
    "ProGen3 is released under a non-commercial license — review the terms at\n"
    f"https://huggingface.co/{PROGEN3_MODEL} before use."
)
try:
    progen3_weights_dir = snapshot_download(repo_id=PROGEN3_MODEL)
    print(f"ProGen3 weights cached at: {progen3_weights_dir}")
except Exception as e:
    print(
        "Could not download ProGen3 automatically. If the model page requires access approval,\n"
        f"visit https://huggingface.co/{PROGEN3_MODEL}, accept the license, then run:\n"
        "  from huggingface_hub import login; login()\n"
        f"Error: {e}"
    )
    raise

TOY_REGRESSION_CSV = os.path.join(TARGET_DIR, "colab_examples", "toy_regression.csv")

print(f"\nCLT checkpoint: {clt_checkpoint}")
print(f"Reference activations: {activations_pt}")
print(f"Toy regression CSV: {TOY_REGRESSION_CSV}")
print("ProGenMech setup complete.")

### 1.5 Discover your own circuit
Use this section to discover a circuit and generate website-ready files.

Select a task mode:
- **CLM**: Provide a single protein prompt sequence. ProGen3 autoregressively generates the continuation and we discover the generation circuit.
- **Zero-shot**: Use the bundled toy regression CSV (or upload your own) and provide sequences to score/visualize.

### **CLM input**
* **Sequence:** A single protein sequence used as the autoregressive prompt.
* **Optional:** `pos` (1-indexed) and `n_generated` to control where generation starts and how many residues are generated.

### **Zero-shot input CSV format (regression only)**
Your CSV must have two specific columns (not case-sensitive):
1. **Sequence**: `sequence` or `mutated_sequence`.
2. **Score**: `score`, `DMS_score`, or `target` (continuous numeric values).

The repo includes a tiny example at `colab_examples/toy_regression.csv` for quick testing.

After discovery, add the sequences you want to score (same length as each other; first sequence is treated as wildtype/reference for differential analysis).

In [ ]:
# @title 1.5 Configure circuit discovery
# @markdown **Settings**

# @markdown Select the task mode:
task_mode = "CLM"  # @param ["CLM", "Zero-shot"]

# @markdown Output folder name:
output_dir = "custom_circuit"  # @param {type:"string"}
# @markdown Where to save the results:
external_path = "/content/experiments"  # @param {type:"string"}
# @markdown Entry name for output JSON:
entry_name = "experiment"  # @param {type:"string"}

# CLM-specific settings
# @markdown CLM prompt sequence (single sequence):
clm_sequence = "MKTAYIAKQRQISFVKSHFSRQ"  # @param {type:"string"}
# @markdown 1-indexed generation start position (leave 0 to append at end):
clm_pos = 0  # @param {type:"integer"}
# @markdown Number of amino acids to generate:
n_generated = 2  # @param {type:"integer"}

# Zero-shot-specific settings
# @markdown Use bundled toy regression CSV (Zero-shot only):
use_bundled_toy_csv = True  # @param {type:"boolean"}
# @markdown Upload your own regression CSV instead (Zero-shot only):
upload_custom_csv = False  # @param {type:"boolean"}

REPO_ROOT = TARGET_DIR
SCRIPT_PATH = os.path.join(REPO_ROOT, "visualization", "auto_discover_circuit_website.py")
EDGE_SCRIPT = os.path.join(REPO_ROOT, "visualization", "get_edge_weights.py")
full_output_dir = os.path.join(external_path, output_dir)
os.makedirs(full_output_dir, exist_ok=True)

DISCOVERY_OUTPUT_DIR = output_dir
DISCOVERY_FULL_OUTPUT_DIR = full_output_dir
DISCOVERY_ENTRY_NAME = entry_name

if task_mode == "CLM":
    if not clm_sequence.strip():
        raise ValueError("CLM mode requires a single prompt sequence.")
    print(f"CLM prompt sequence length: {len(clm_sequence.strip())}")
else:
    print("Zero-shot mode selected. Run the next cell to add sequences, then run discovery.")

In [ ]:
# @title 2. Add sequences to score (Zero-shot only; run after section 1.5)
# @markdown Example sequences from `colab_examples/toy_regression.csv`:
# @markdown - Seq 1 (wildtype): `MKTAYIAKQRQISFVKSHFSRQ`
# @markdown - Seq 2: `MKTGYIAKQRQISFVKSHFSRQ`
# @markdown - Seq 3: `MKTAYIEKQRQISFVKSHFSRQ`
# @markdown
# @markdown Note: Use `Add Sequence +` to compare variations of the same protein. For CLM mode, skip this cell and run discovery directly.

TOY_SEQUENCES = [
    "MKTAYIAKQRQISFVKSHFSRQ",
    "MKTGYIAKQRQISFVKSHFSRQ",
    "MKTAYIEKQRQISFVKSHFSRQ",
]

class SequenceInputManager:
    def __init__(self, initial_sequences=None):
        self.sequences = []
        self.container = widgets.VBox()
        self.add_button = widgets.Button(description="Add Sequence +", icon="plus")
        self.add_button.on_click(self.add_field)
        if initial_sequences:
            for seq in initial_sequences:
                self.add_field(None, default_value=seq)
        else:
            self.add_field(None)

    def add_field(self, b, default_value=""):
        idx = len(self.sequences) + 1
        label = "Seq 1 (wildtype):" if idx == 1 else f"Seq {idx}:"
        text = widgets.Text(
            value=default_value,
            placeholder=f"Enter protein sequence {idx}...",
            layout=widgets.Layout(width='80%'),
        )
        box = widgets.HBox([widgets.Label(value=label, layout=widgets.Layout(width='120px')), text])
        self.sequences.append(text)
        self.container.children = tuple(list(self.container.children) + [box])

    def get_sequences(self):
        return [w.value.strip() for w in self.sequences if w.value.strip()]

    def display(self):
        display(widgets.VBox([self.container, self.add_button]))

seq_manager = SequenceInputManager(initial_sequences=TOY_SEQUENCES)
seq_manager.display()

In [ ]:
# @title 3. Generate circuit files (run after configuring task and adding sequences)
# @markdown Runs `auto_discover_circuit_website.py` and then computes `virtual_weights.json`.

def run_realtime(cmd):
    master, slave = pty.openpty()
    p = subprocess.Popen(cmd, stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    try:
        while True:
            try:
                data = os.read(master, 1024).decode()
                if not data:
                    break
                sys.stdout.write(data)
                sys.stdout.flush()
            except OSError:
                break
    except Exception:
        pass
    p.wait()
    os.close(master)
    return p.returncode


def resolve_zero_shot_csv():
    if use_bundled_toy_csv:
        bundled = globals().get("TOY_REGRESSION_CSV", os.path.join(REPO_ROOT, "colab_examples", "toy_regression.csv"))
        if not os.path.exists(bundled):
            raise FileNotFoundError(
                f"Bundled toy CSV not found at {bundled}. "
                "Re-run setup or set use_bundled_toy_csv=False to upload your own file."
            )
        print(f"Using bundled toy regression CSV: {bundled}")
        return bundled
    if upload_custom_csv:
        if not IN_COLAB:
            raise RuntimeError("Custom CSV upload requires Google Colab.")
        print("Please upload your regression CSV file...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Upload cancelled.")
        csv_name = list(uploaded.keys())[0]
        return os.path.join(os.getcwd(), csv_name)
    raise RuntimeError("For Zero-shot, enable use_bundled_toy_csv or upload_custom_csv.")


cmd = [
    sys.executable, SCRIPT_PATH,
    "--output_dir", full_output_dir,
    "--entry_name", entry_name,
    "--max_nodes", "1000",
    "--step_size", "32",
]

if task_mode == "CLM":
    cmd += ["--type", "CLM", "--seq", clm_sequence.strip(), "--n_generated", str(n_generated)]
    if clm_pos and clm_pos > 0:
        cmd += ["--pos", str(clm_pos)]
else:
    csv_path = resolve_zero_shot_csv()
    cmd += ["--type", "zero_shot", "--csv_path", csv_path]
    sequences = seq_manager.get_sequences()
    if sequences:
        lengths = [len(s) for s in sequences]
        if len(set(lengths)) != 1:
            raise ValueError("All sequences must have the same length.")
        cmd += ["--sequences"] + sequences

print(f"\nStarting Discovery ({task_mode})...")
print(f"   Output: {full_output_dir}/{entry_name}.json\n")
exit_code = run_realtime(cmd)

if exit_code != 0:
    print("\nDiscovery Failed.")
else:
    print("\n[Step 2] Computing virtual edge weights (may take some time)...")
    folders = []
    if os.path.isdir(full_output_dir):
        for name in sorted(os.listdir(full_output_dir)):
            path = os.path.join(full_output_dir, name)
            if os.path.isdir(path) and name.startswith("seq"):
                folders.append(path)
    has_root_inputs = (
        os.path.exists(os.path.join(full_output_dir, "generation.fasta"))
        or os.path.exists(os.path.join(full_output_dir, "seq.txt"))
    )
    if has_root_inputs:
        folders.insert(0, full_output_dir)

    for folder_path in folders:
        folder_name = os.path.basename(folder_path)
        print(f"\n   Processing {folder_name}...")
        weight_cmd = [
            sys.executable, EDGE_SCRIPT,
            "--base_folder", folder_path,
            "--clt_ckpt", clt_checkpoint,
        ]
        w_exit = run_realtime(weight_cmd)
        if w_exit != 0:
            print(f"Failed to compute weights for {folder_name}")

    print("\n==========")
    print(f"Results saved to: {full_output_dir}")
    print("==========")
    print("Files generated:")
    for f in sorted(os.listdir(full_output_dir)):
        suffix = "/" if os.path.isdir(os.path.join(full_output_dir, f)) else ""
        print(f" - {f}{suffix}")